In [1]:
import torch
from torch.functional import F

class MLPClient(torch.nn.Module):
    def __init__(self, hidden_size: tuple[int, int] = (200, 100)):
        super(MLPClient, self).__init__()
        self.fc1 = torch.nn.Linear(28 * 28, hidden_size[0])
        self.fc2 = torch.nn.Linear(hidden_size[0], hidden_size[1])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x


class MLPServer(torch.nn.Module):
    def __init__(self, hidden_size: int = 100, use_softmax: bool = False):
        super(MLPServer, self).__init__()
        self.use_softmax = use_softmax
        self.fc3 = torch.nn.Linear(hidden_size, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_softmax:
            return F.softmax(self.fc3(x), dim=1)
        else:
            return self.fc3(x)



In [2]:
from fluke.data.datasets import Datasets
from fluke.data import DataSplitter


dataset = Datasets.get("mnist", path="./data")

splitter = DataSplitter(dataset=dataset,
                        distribution="iid")

In [3]:
from fluke.evaluation import ClassificationEval
from fluke import FlukeENV, DDict

env = FlukeENV()
env.set_seed(42) # we set a seed for reproducibility
env.set_device("cpu") # we use the CPU for this example
env.set_evaluator(ClassificationEval(eval_every=1, n_classes=dataset.num_classes))

client_hp = DDict(
    batch_size=10,
    local_epochs=3,
    loss="CrossEntropyLoss",
    optimizer=DDict(
        lr=0.01,
        momentum=0.9,
        weight_decay=0.0001),
    scheduler=DDict(
        gamma=1,
        step_size=1)
)

server_hp = DDict(
    loss="CrossEntropyLoss",
    optimizer=DDict(
        lr=0.01,
        momentum=0.9,
        weight_decay=0.0001),
    scheduler=DDict(
        gamma=1,
        step_size=1)
)

hyper_params = DDict(client=client_hp,
                    server=server_hp,
                    model={"client": MLPClient(), "server": MLPServer()})


In [4]:
from fluke.utils.log import Log
from playground.vanillaSL import VanillaSL

algorithm = VanillaSL(n_clients=10, # 10 clients in the federation
                   data_splitter=splitter,
                   hyper_params=hyper_params)

logger = Log()
algorithm.set_callbacks(logger)

In [5]:
algorithm.run(n_rounds=3, eligible_perc=1)

Output()

╭────────────────────────────────────── Overall Performance ───────────────────────────────────────╮
│ {                                                                                                │
│     'global': {                                                                                  │
│         'accuracy': 0.9809,                                                                      │
│         'macro_precision': 0.98091,                                                              │
│         'macro_recall': 0.98064,                                                                 │
│         'macro_f1': 0.98075,                                                                     │
│         'micro_precision': 0.9809,                                                               │
│         'micro_recall': 0.9809,                                                                  │
│         'micro_f1': 0.9809,                                                                      │
│         'round': 3                                                                               │
│     },                                                                                           │
│     'comm_costs': 10626000                                                                       │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯